# A Modern Training Loop: CIFAR10, Mixed Precision, torch.compile

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/05-pytorch/03-Modern-Training-Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/05-pytorch/'
for fn in ['q_dataset.py', 'q_amp_benchmark.py', 'q_tta.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from IPython.display import display
device = "cuda" if torch.cuda.is_available() else "cpu"
import os
NUM_WORKERS = 2 if os.name != "nt" else 0   # Colab (Linux) uses 2 workers; set 0 when running locally on Windows
torch.backends.cudnn.benchmark = True   # with fixed input sizes, let cuDNN pick the fastest algorithm

## CIFAR10

Ten classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck), 32x32 color images.

In [ ]:
# Download from the Hugging Face Hub (much faster than the original CIFAR site)
from datasets import load_dataset
from types import SimpleNamespace
def load_cifar10(split):
    ds = load_dataset("uoft-cs/cifar10", split=split)
    return SimpleNamespace(data=np.stack([np.array(im) for im in ds["img"]]), targets=list(ds["label"]))
train_raw = load_cifar10("train")
test_raw = load_cifar10("test")
name_array = np.array(["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"])
train_raw.data.shape, len(train_raw.targets)

In [ ]:
def showX(X):
    """X: (N, 32, 32, 3) uint8"""
    display(Image.fromarray(np.concatenate(list(X), axis=1)).resize((64 * len(X), 64), Image.NEAREST))
showX(train_raw.data[:12])
print(name_array[train_raw.targets[:12]])

## Dataset and DataLoader

A `Dataset` only needs to implement `__len__` and `__getitem__`; the `DataLoader` handles batching, shuffling, and parallel loading with multiple workers.

`torchvision.transforms.v2` is the new image transform API.

In [ ]:
import torchvision.transforms.v2 as T
mean = np.array([0.4914, 0.4822, 0.4465])
std = np.array([0.2470, 0.2435, 0.2616])
train_tf = T.Compose([
    T.ToImage(),
    T.RandomCrop(32, padding=4, padding_mode="reflect"),
    T.RandomHorizontalFlip(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean, std),
])
train_tf(train_raw.data[0]).shape

## Q
Write your own `Dataset` class that wraps `train_raw.data`, `train_raw.targets`, and a `transform`, then use a `DataLoader` to read one batch and check its shape.

In [ ]:
# %load q_dataset.py
%run -i q_dataset.py

### Keeping the data on the GPU

Colab has only 2 CPU cores, so doing data augmentation on the CPU often becomes the bottleneck, leaving the GPU waiting for data.
CIFAR10 is small (50000x3x32x32), so we can put all of it on the GPU and do the augmentation there too, which is much faster.

In [ ]:
def to_tensor(raw):
    X = torch.tensor(raw.data).permute(0, 3, 1, 2).float().div(255)   # NHWC -> NCHW
    X = (X - torch.tensor(mean).view(1, 3, 1, 1).float()) / torch.tensor(std).view(1, 3, 1, 1).float()
    return X.to(device), torch.tensor(raw.targets).to(device)
train_X, train_y = to_tensor(train_raw)
test_X, test_y = to_tensor(test_raw)

def gpu_augment(X, pad=4):
    """random crop (reflect padding) + random horizontal flip, processing the whole batch at once on the GPU"""
    N, C, H, W = X.shape
    Xp = F.pad(X, (pad,) * 4, mode="reflect")
    dx, dy = torch.randint(0, 2 * pad + 1, (2, N), device=X.device)
    rows = (torch.arange(H, device=X.device)[None, :] + dy[:, None])        # N x H
    cols = (torch.arange(W, device=X.device)[None, :] + dx[:, None])        # N x W
    X = Xp[torch.arange(N, device=X.device)[:, None, None, None],
           torch.arange(C, device=X.device)[None, :, None, None],
           rows[:, None, :, None], cols[:, None, None, :]]
    flip = torch.rand(N, device=X.device) < 0.5
    return torch.where(flip[:, None, None, None], X.flip(3), X)

def batches(X, y, batch_size, shuffle=False, augment=False):
    idx = torch.randperm(len(X), device=X.device) if shuffle else torch.arange(len(X), device=X.device)
    for i in range(0, len(X), batch_size):
        j = idx[i:i + batch_size]
        xb = X[j]
        yield (gpu_augment(xb) if augment else xb), y[j]

In [ ]:
xb, yb = next(batches(train_X, train_y, 12, augment=True))
img = (xb.permute(0, 2, 3, 1).cpu().numpy() * std + mean).clip(0, 1)
showX((img * 255).astype(np.uint8))

## Model: a small ResNet

* BatchNorm stabilizes training
* Residual connections make deep networks trainable

In [ ]:
def conv_bn(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.GELU(),
    )

class Residual(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.block = nn.Sequential(conv_bn(c, c), conv_bn(c, c))
    def forward(self, x):
        return x + self.block(x)

def resnet9(num_classes=10, w=64):
    return nn.Sequential(
        conv_bn(3, w),
        conv_bn(w, 2 * w), nn.MaxPool2d(2), Residual(2 * w),      # 16x16
        conv_bn(2 * w, 4 * w), nn.MaxPool2d(2),                   # 8x8
        conv_bn(4 * w, 8 * w), nn.MaxPool2d(2), Residual(8 * w),  # 4x4
        nn.AdaptiveMaxPool2d(1), nn.Flatten(),
        nn.Linear(8 * w, num_classes),
    )
model = resnet9()
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## Mixed precision training (Automatic Mixed Precision, AMP)

* `torch.autocast`: automatically switches suitable ops (matmul, conv) to fp16 and keeps the rest (e.g. softmax, loss) in fp32
* `GradScaler`: fp16 has a small range, so very small gradients become 0 (underflow). It scales the loss up before backward and scales the gradients back down before the update
* On A100/H100/RTX 30-series or newer cards you can use `dtype=torch.bfloat16` instead and skip the GradScaler. **The T4 only supports fp16**

In [ ]:
@torch.no_grad()
def evaluate(model, X=None, y=None, amp=True):
    X = test_X if X is None else X
    y = test_y if y is None else y
    model.eval()
    correct = 0
    for xb, yb in batches(X, y, 1000):
        with torch.autocast(device, dtype=torch.float16, enabled=amp):
            correct += (model(xb.contiguous(memory_format=torch.channels_last)).argmax(1) == yb).sum().item()
    return correct / len(X)

def fit(model, epochs=10, batch_size=512, lr=2e-3, amp=True, compile=False, label_smoothing=0.1):
    model = model.to(device, memory_format=torch.channels_last)   # channels_last is faster for Tensor Core convolutions
    train_model = torch.compile(model) if compile else model
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-4, fused=(device == "cuda"))
    steps_per_epoch = (len(train_X) + batch_size - 1) // batch_size
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs * steps_per_epoch)
    scaler = torch.amp.GradScaler(device, enabled=amp)
    for epoch in range(epochs):
        model.train()
        t0 = time.perf_counter()
        total_loss = 0
        for xb, yb in batches(train_X, train_y, batch_size, shuffle=True, augment=True):
            xb = xb.contiguous(memory_format=torch.channels_last)
            with torch.autocast(device, dtype=torch.float16, enabled=amp):
                loss = F.cross_entropy(train_model(xb), yb, label_smoothing=label_smoothing)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()
        if device == "cuda": torch.cuda.synchronize()
        dt = time.perf_counter() - t0
        print(f"epoch {epoch:2d}  loss={total_loss/steps_per_epoch:.3f}  test acc={evaluate(model, amp=amp):.4f}"
              f"  {dt:.1f}s ({len(train_X)/dt:.0f} img/s)")
    return model

In [ ]:
model = fit(resnet9(), epochs=10)

10 epochs should reach about 90% accuracy.

In [ ]:
# Look at the predictions
with torch.no_grad(), torch.autocast(device, dtype=torch.float16):
    pred = model(test_X[:12]).argmax(1).cpu()
showX(test_raw.data[:12])
print("pred:", name_array[pred])
print("label:", name_array[test_raw.targets[:12]])

## Q
Compare the following settings, running only 2 epochs each, and see how many images per second (img/s) they process:
* `amp=False` (pure fp32)
* `amp=True`
* `amp=True, compile=True` (the first epoch includes compile time, so look at the second epoch)

`torch.compile` compiles the model into optimized kernels (using Triton), so the first run is slower.

Note: **always measure** optimizations. On the T4, AMP is roughly 3-4x faster, but `torch.compile` may actually slow down this small model
(cuDNN convolutions are already fast, and the T4 doesn't have enough SMs for some autotuning modes). Compile usually helps more with large models or models with many small elementwise ops.

In [ ]:
# %load q_amp_benchmark.py
%run -i q_amp_benchmark.py

## Saving a checkpoint

To resume training later, save the optimizer, scheduler, and scaler states in addition to the model parameters.
Here we only show the basics: saving the model parameters.

In [ ]:
torch.save(model.state_dict(), "resnet9_cifar10.pt")
model2 = resnet9().to(device, memory_format=torch.channels_last)
model2.load_state_dict(torch.load("resnet9_cifar10.pt"))
evaluate(model2)

## Q: Test Time Augmentation (TTA)
At prediction time, feed both the original image and its horizontal flip into the model and average the two outputs (logits or probabilities). Does accuracy improve?

In [ ]:
# %load q_tta.py
%run -i q_tta.py